# Giai đoạn 1: Data Collection (Cào Dữ Liệu)

Notebook này sinh 2 file CSV trong `Notebook_Report/`:
- `cinesense_movies.csv`
- `cinesense_reviews.csv`

Mặc định notebook sẽ **cào TMDB từ web** (qua API) và lưu CSV đúng schema mà notebook 02 dùng.

## Tham số cào (bạn có thể chỉnh để cân bằng thời gian)
- `MAX_MOVIES`: số lượng phim cần thu thập (mặc định 4900)
- `MOVIE_ENDPOINT`: endpoint danh sách phim (`/movie/popular`)
- `MOVIE_PAGES_END`: số trang tương ứng ~ `MAX_MOVIES / 20` (mặc định tự tính)
- `MAX_REVIEWS_PER_MOVIE`: giới hạn số review mỗi phim (mặc định 2 để tiệm cận ~9k reviews)
- `REVIEW_PAGES_PER_MOVIE`: số trang tối đa của endpoint reviews mỗi phim (mặc định 5)

> Nếu bạn muốn chạy nhanh mà không cào, bạn có thể đổi sang cơ chế nạp snapshot (đã có sẵn trong code hiện tại) bằng cách set flag ở cell code.

In [2]:
import pandas as pd
from pathlib import Path

# Notebook chạy từ thư mục `Notebook_Report/`.
DATA_DIR = Path(".")
movies_path = DATA_DIR / "cinesense_movies.csv"
reviews_path = DATA_DIR / "cinesense_reviews.csv"

# Mục tiêu: tạo ra 2 file CSV như “kết quả của TMDB crawl”.
# Trong report/demo, để không tốn thời gian gọi TMDB, mình nạp lại snapshot đã export sẵn từ Postgres
# (snapshot này vốn được TMDB nạp vào).

def load_snapshot(max_movies: int | None = None, max_reviews_per_movie: int | None = None):
    if not movies_path.exists() or not reviews_path.exists():
        raise FileNotFoundError(
            "Chưa có cinesense_movies.csv/cinesense_reviews.csv trong Notebook_Report/.\n"
            "Bạn có thể xuất nhanh từ Postgres bằng script: "
            "python3 scripts/export_dataset_csv_from_db.py --out-dir Notebook_Report --only-english"
        )

    df_movies = pd.read_csv(movies_path).fillna("")
    df_reviews = pd.read_csv(reviews_path).fillna("")

    # Chuẩn hoá kiểu dữ liệu để các notebook sau merge/group đúng.
    if "tmdb_id" in df_movies.columns:
        df_movies["tmdb_id"] = df_movies["tmdb_id"].astype(int)
    if "tmdb_id" in df_reviews.columns:
        df_reviews["tmdb_id"] = df_reviews["tmdb_id"].astype(int)

    if max_movies is not None:
        df_movies = df_movies.head(max_movies)
        keep_ids = set(df_movies["tmdb_id"].tolist())
        df_reviews = df_reviews[df_reviews["tmdb_id"].isin(keep_ids)].copy()

    if max_reviews_per_movie is not None:
        # Giữ ổn định: nếu có created_at thì sort trước, còn không thì giữ nguyên thứ tự.
        if "created_at" in df_reviews.columns:
            df_reviews = df_reviews.replace({"created_at": {"": pd.NA}})
            # `errors='coerce'` để tránh crash khi created_at có giá trị rỗng
            df_reviews["_created_at"] = pd.to_datetime(df_reviews["created_at"], errors="coerce")
            df_reviews = df_reviews.sort_values(["tmdb_id", "_created_at"], ascending=[True, True])
            df_reviews = df_reviews.groupby("tmdb_id", group_keys=False).head(max_reviews_per_movie)
            df_reviews = df_reviews.drop(columns=["_created_at"]) if "_created_at" in df_reviews.columns else df_reviews
        else:
            df_reviews = df_reviews.groupby("tmdb_id", group_keys=False).head(max_reviews_per_movie)

    # Ghi lại để các notebook 02→05 dùng đúng output như thiết kế của report.
    df_movies.to_csv("cinesense_movies.csv", index=False)
    df_reviews.to_csv("cinesense_reviews.csv", index=False)

    return df_movies, df_reviews

# Thiết lập theo báo cáo
max_movies = 5000
max_reviews_per_movie = 5

try:
    df_movies, df_reviews = load_snapshot(max_movies=max_movies, max_reviews_per_movie=max_reviews_per_movie)

    print(f"\nGiả lập TMDB crawl (từ snapshot Postgres): {len(df_movies)} movies và {len(df_reviews)} reviews.")

    print("\nMẫu file Movies:")
    display(df_movies.head(2))

    print("\nMẫu file Reviews:")
    display(df_reviews.head(2))
except FileNotFoundError as e:
    # Nếu chưa có snapshot csv, không chặn cell crawl phía dưới.
    print(str(e))
    print("Bỏ qua snapshot vì sẽ chạy TMDB crawl ở cell 2.")
    df_movies = pd.DataFrame()
    df_reviews = pd.DataFrame()


Bắt đầu cào dữ liệu từ TMDB...
Thu thập được 60 movies và 134 reviews.

Mẫu file Movies:


,tmdb_id,title,original_title,overview,genres,release_date,poster_path,backdrop_path,vote_average,vote_count,popularity
0,875828,Peaky Blinders: The Immortal Man,Peaky Blinders: The Immortal Man,After his estranged son gets embroiled in a Na...,"Crime, Drama",2026-03-05,/gRMalasZEzsZi4w2VFuYusfSfqf.jpg,/1fkuBPid72KGS6WmtkEXMftZtkE.jpg,7.401,408,354.7335
1,83533,Avatar: Fire and Ash,Avatar: Fire and Ash,In the wake of the devastating war against the...,"Science Fiction, Adventure, Fantasy",2025-12-17,/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg,/sdZSjtGUTSN8B3al5o0f2WoQfQQ.jpg,7.300,1927,322.8645



Mẫu file Reviews:


,review_id,tmdb_id,author,author_name,content,rating,avatar_path,created_at,url
0,69b2a90fab3233b77a685bd7,875828,CinemaSerf,CinemaSerf,Anyone remember Michael Elphick’s “Private Sch...,7.0,/9HcQx0Yfbxy8eqr3ft66X9uWMf0.jpg,2026-03-12T11:52:47.670Z,https://www.themoviedb.org/review/69b2a90fab32...
1,6943d8d0a56edf751f705664,83533,Manuel São Bento,Manuel São Bento,FULL SPOILER-FREE REVIEW @ https://movieswetex...,5.0,NaN,2025-12-18T10:34:56.770Z,https://www.themoviedb.org/review/6943d8d0a56e...


In [ ]:
import os
import math
import time

import httpx
import pandas as pd

TMDB_API_KEY = os.getenv("TMDB_API_KEY", "")
TMDB_BASE_URL = "https://api.themoviedb.org/3"

# =========================
# Tham số cào (chỉnh nếu cần)
# =========================
MAX_MOVIES = 4900
MOVIE_ENDPOINT = "/movie/popular"  # có thể đổi sang /movie/top_rated nếu muốn
MOVIE_PAGES_START = 1
MOVIE_PAGE_SIZE = 20  # TMDB trả mặc định 20 results/page
MOVIE_PAGES_END = MOVIE_PAGES_START + math.ceil(MAX_MOVIES / MOVIE_PAGE_SIZE) - 1

MAX_REVIEWS_PER_MOVIE = 2  # để ~9k reviews (xấp xỉ)
REVIEW_PAGES_PER_MOVIE = 5

ONLY_ENGLISH_REVIEWS = True
LANG_WHITELIST = {"en", "unknown", None}

SLEEP_BETWEEN_MOVIES_S = 0.12
SLEEP_BETWEEN_REVIEWS_S = 0.05
TIMEOUT_S = 20.0

OUT_MOVIES = "cinesense_movies.csv"
OUT_REVIEWS = "cinesense_reviews.csv"

if not TMDB_API_KEY:
    raise ValueError(
        "TMDB_API_KEY chưa được set. Vui lòng export biến môi trường trước khi chạy notebook, ví dụ:\n"
        "export TMDB_API_KEY='<key-của-bạn>'"
    )

print("=== TMDB Crawl Configuration ===")
print(f"MAX_MOVIES={MAX_MOVIES}")
print(f"MOVIE_ENDPOINT={MOVIE_ENDPOINT}")
print(f"MOVIE_PAGES_END={MOVIE_PAGES_END} (ước tính từ MAX_MOVIES/20)")
print(f"MAX_REVIEWS_PER_MOVIE={MAX_REVIEWS_PER_MOVIE}")
print(f"REVIEW_PAGES_PER_MOVIE={REVIEW_PAGES_PER_MOVIE}")

client = httpx.Client(base_url=TMDB_BASE_URL, timeout=TIMEOUT_S)
movies: list[dict] = []
reviews: list[dict] = []

try:
    # 1) Genres mapping
    genres_dict: dict[int, str] = {}
    g_resp = client.get(
        "/genre/movie/list",
        params={"api_key": TMDB_API_KEY, "language": "en-US"},
    )
    g_resp.raise_for_status()
    for g in g_resp.json().get("genres", []) or []:
        if "id" in g and "name" in g:
            genres_dict[int(g["id"])] = str(g["name"])

    # 2) Fetch movies + reviews
    for page in range(MOVIE_PAGES_START, MOVIE_PAGES_END + 1):
        if len(movies) >= MAX_MOVIES:
            break

        m_resp = client.get(
            MOVIE_ENDPOINT,
            params={"api_key": TMDB_API_KEY, "language": "en-US", "page": page},
        )
        m_resp.raise_for_status()
        items = m_resp.json().get("results", []) or []

        for item in items:
            if len(movies) >= MAX_MOVIES:
                break

            movie_id = item.get("id")
            if movie_id is None:
                continue

            genre_ids = item.get("genre_ids") or []
            movie_genres = [genres_dict.get(gid, "Unknown") for gid in genre_ids]

            movies.append(
                {
                    "tmdb_id": int(movie_id),
                    "title": item.get("title") or item.get("original_title") or "",
                    "original_title": item.get("original_title") or item.get("title") or "",
                    "overview": item.get("overview") or "",
                    "genres": ", ".join(movie_genres),
                    "release_date": item.get("release_date") or "",
                    "poster_path": item.get("poster_path") or "",
                    "backdrop_path": item.get("backdrop_path") or "",
                    "vote_average": item.get("vote_average", ""),
                    "vote_count": item.get("vote_count", ""),
                    "popularity": item.get("popularity", ""),
                }
            )

            # --- Reviews ---
            got = 0
            for rp in range(1, REVIEW_PAGES_PER_MOVIE + 1):
                if got >= MAX_REVIEWS_PER_MOVIE:
                    break

                r_resp = client.get(
                    f"/movie/{movie_id}/reviews",
                    params={"api_key": TMDB_API_KEY, "page": rp},
                )

                # Rate limit protection
                if r_resp.status_code == 429:
                    retry_after = int(r_resp.headers.get("Retry-After", "2"))
                    print(f"429 on reviews (movie_id={movie_id}); sleeping {retry_after}s")
                    time.sleep(retry_after)
                    r_resp = client.get(
                        f"/movie/{movie_id}/reviews",
                        params={"api_key": TMDB_API_KEY, "page": rp},
                    )

                r_resp.raise_for_status()
                r_items = r_resp.json().get("results", []) or []
                if not r_items:
                    break

                for r in r_items:
                    if got >= MAX_REVIEWS_PER_MOVIE:
                        break

                    lang = r.get("language")
                    if ONLY_ENGLISH_REVIEWS and lang not in LANG_WHITELIST:
                        continue

                    author_details = r.get("author_details") or {}
                    author = r.get("author") or "Anonymous"
                    author_name = author_details.get("name") or r.get("author_name") or author

                    reviews.append(
                        {
                            "review_id": r.get("id") or "",
                            "tmdb_id": int(movie_id),
                            "author": author,
                            "author_name": author_name,
                            "content": r.get("content") or "",
                            "rating": author_details.get("rating", ""),
                            "avatar_path": author_details.get("avatar_path") or "",
                            "created_at": r.get("created_at") or "",
                            "url": r.get("url") or "",
                            "source": "tmdb",
                        }
                    )
                    got += 1

                    if SLEEP_BETWEEN_REVIEWS_S:
                        time.sleep(SLEEP_BETWEEN_REVIEWS_S)

            if SLEEP_BETWEEN_MOVIES_S:
                time.sleep(SLEEP_BETWEEN_MOVIES_S)

            if len(movies) >= MAX_MOVIES:
                break

        print(f"Progress: page={page} movies={len(movies)} reviews={len(reviews)}")

finally:
    client.close()

print(f"\nCrawl done: movies={len(movies)}, reviews={len(reviews)}")

# 3) Save CSV
movies_df = pd.DataFrame(movies)
reviews_df = pd.DataFrame(reviews)

movies_df.to_csv(OUT_MOVIES, index=False, encoding="utf-8", na_rep="")
reviews_df.to_csv(OUT_REVIEWS, index=False, encoding="utf-8", na_rep="")

print(f"Saved: {OUT_MOVIES} ({len(movies_df)} rows)")
print(f"Saved: {OUT_REVIEWS} ({len(reviews_df)} rows)")

display(movies_df.head(2))
display(reviews_df.head(2))
